# 7. Cherry-pick

`git cherry-pick` copies **one commit** onto the branch you are on.

When is that useful? The barge software was delivered to a customer from
a `release` branch. Then a bug was found and fixed on `main` — but `main`
also has new, untested features. The customer needs the fix, and only the
fix.

```mermaid
gitGraph
    commit id: "volume + froude (bug)"
    branch release
    checkout main
    commit id: "fix"
    commit id: "tonnes per cm"
    commit id: "wetted surface"
    checkout release
    cherry-pick id: "fix"
```

In [ ]:
!rm -rf ~/git-course/07-cherry-pick
!mkdir -p ~/git-course/07-cherry-pick
%cd ~/git-course/07-cherry-pick

## Starting point

The delivered code. The Froude number is missing a square root:

In [ ]:
%%writefile hydrostatics.py
"""Hydrostatics for a box-shaped barge."""

RHO = 1025.0  # kg/m^3, sea water


def volume(length, beam, draft):
    """Immersed volume of the hull, in m^3."""
    return length * beam * draft


def froude_number(speed, length):
    """Speed-length ratio that scales wave resistance."""
    return speed / (9.81 * length)

In [ ]:
%%writefile test_hydrostatics.py
from hydrostatics import froude_number

# An 80 m hull at 12 knots (6.17 m/s) should have Fn of about 0.22.
assert 0.21 < froude_number(6.17, 80.0) < 0.23

print("all checks passed")

In [ ]:
!git init
!git add hydrostatics.py test_hydrostatics.py
!git commit -m "Add volume and the Froude number"

This is the delivered version, so we put a `release` branch here:

In [ ]:
!git branch release

## Work continues on main

First, the fix:

In [ ]:
%%writefile hydrostatics.py
"""Hydrostatics for a box-shaped barge."""

RHO = 1025.0  # kg/m^3, sea water


def volume(length, beam, draft):
    """Immersed volume of the hull, in m^3."""
    return length * beam * draft


def froude_number(speed, length):
    """Speed-length ratio that scales wave resistance."""
    return speed / (9.81 * length) ** 0.5

In [ ]:
!git add hydrostatics.py
!git commit -m "Fix the Froude number: take the square root"

Then two new features:

In [ ]:
%%writefile -a hydrostatics.py


def tonnes_per_cm(length, beam):
    """Tonnes of cargo that sink the barge by one centimetre."""
    return length * beam * RHO / 100000.0

In [ ]:
!git add hydrostatics.py
!git commit -m "Add tonnes per centimetre immersion"

In [ ]:
%%writefile -a hydrostatics.py


def wetted_surface(length, beam, draft):
    """Rough wetted surface area, in m^2."""
    return length * (beam + 2 * draft)

In [ ]:
!git add hydrostatics.py
!git commit -m "Add a rough wetted surface estimate"

In [ ]:
!git log --oneline --graph --all

## The release still has the bug

In [ ]:
!git switch release

In [ ]:
# This one fails on purpose: the bug the customer reported.
!python test_hydrostatics.py

## Picking the fix

Find the fix in the log of `main`:

In [ ]:
!git log --oneline main

Normally you copy its id (the short hash) from there. The id is different
on every computer, so in this notebook we name it `main~2` instead: "two
commits before the end of `main`".

In [ ]:
!git cherry-pick main~2

In [ ]:
!python test_hydrostatics.py

In [ ]:
!git log --oneline --graph --all

The release is fixed, and it did **not** get the two new features. The fix
now exists twice, as two commits with different ids: cherry-pick copies.

## A conflict

Cherry-picking can conflict like a merge. Let's try to pick the wetted
surface feature, which was written after `tonnes_per_cm`:

In [ ]:
# This one fails on purpose: a conflict.
!git cherry-pick main

We don't want that feature in the release anyway, so we give up:

In [ ]:
!git cherry-pick --abort
!git status

## When to use it

- **Yes:** copying a bug fix to a release branch.
- **Yes:** moving a commit you made on the wrong branch.
- **No:** instead of merging a whole branch. Use `git merge` for that.

## Summary

| I want to… | Command |
| - | - |
| copy one commit here | `git cherry-pick <commit>` |
| give up after a conflict | `git cherry-pick --abort` |
| continue after fixing a conflict | `git cherry-pick --continue` |

## Exercises

1. Make a commit on the wrong branch, then copy it to the right one with
   cherry-pick.
2. Mark the fixed release with a tag: `git tag v1.0.1`, then
   `git log --oneline`.

Next: [8. Submodules](08-submodules.ipynb)